In [85]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [86]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [4]:
!wget -P /home/asus/Machine-Learning/ml-zoomcamp-2k26/module_3_classification/datasets/ $data

--2026-10-11 12:01:20--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘/home/asus/Machine-Learning/ml-zoomcamp-2k26/module_3_classification/datasets/course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.1s    

2026-10-11 12:01:21 (1.84 MB/s) - ‘/home/asus/Machine-Learning/ml-zoomcamp-2k26/module_3_classification/datasets/course_lead_scoring_2026.csv’ saved [278746/278746]



In [87]:
df = pd.read_csv("/home/asus/Machine-Learning/ml-zoomcamp-2k26/module_3_classification/datasets/course_lead_scoring_2026.csv")

In [88]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [89]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [90]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [91]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical = list(df.columns[df.dtypes == 'str'])
numerical = list(df.columns[df.dtypes != 'str'])

In [92]:
df[categorical] = df[categorical].fillna('unknown')
df[numerical] = df[numerical].fillna(0)

In [93]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [94]:
df['lead_source'].unique()

<StringArray>
['organic_search', 'social_media', 'referral', 'paid_ads', 'unknown',
 'events']
Length: 6, dtype: str

In [95]:


for c in categorical:
    df[c] = df[c].str.lower().str.replace(' ', '_')
for c in categorical:
    df[c] = df[c].str.replace('unknown', 'NA')

In [96]:
df['lead_source'].unique()

<StringArray>
['organic_search', 'social_media', 'referral', 'paid_ads', 'NA', 'events']
Length: 6, dtype: str

In [97]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NA,north_america
annual_income,88160.0,72688.0,44697.0,0.0,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [98]:
# can see from above result that data cleaning is done... although there was nothing to clean...lol
# our target variable is "converted"

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 1)
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 1)

for c in (df_train, df_val, df_test) : print(len(c))

3000
1000
1000


In [99]:
# GTR line

df_train, df_val, df_test = map(lambda df: df.reset_index(drop = True), (df_train, df_val, df_test))

In [100]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,technology,employed,asia,80198.0,5,9,0.91,1
1,paid_ads,retail,employed,asia,50548.0,2,5,0.48,0
2,NA,finance,employed,NA,76300.0,0,3,0.42,0
3,paid_ads,finance,employed,NA,80371.0,2,6,0.49,0
4,organic_search,finance,employed,asia,73567.0,2,4,0.48,1


In [101]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train, df_val, df_test = [df.drop(columns = ['converted']) for df in (df_train, df_val, df_test)]

In [102]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,technology,employed,asia,80198.0,5,9,0.91
1,paid_ads,retail,employed,asia,50548.0,2,5,0.48
2,NA,finance,employed,NA,76300.0,0,3,0.42
3,paid_ads,finance,employed,NA,80371.0,2,6,0.49
4,organic_search,finance,employed,asia,73567.0,2,4,0.48


In [103]:
df_full_train.reset_index(drop=True)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,technology,student,europe,38757.0,0,0,0.15,0
1,paid_ads,healthcare,self_employed,north_america,56516.0,2,6,0.59,1
2,paid_ads,retail,employed,NA,54963.0,1,0,0.13,0
3,social_media,retail,self_employed,NA,26237.0,1,5,0.46,0
4,organic_search,retail,student,north_america,21992.0,3,6,0.70,1
...,...,...,...,...,...,...,...,...,...
3995,paid_ads,manufacturing,NA,north_america,66121.0,1,1,0.20,0
3996,paid_ads,manufacturing,employed,north_america,60720.0,4,6,0.69,1
3997,NA,manufacturing,NA,NA,49820.0,2,3,0.44,0
3998,organic_search,manufacturing,self_employed,north_america,52228.0,3,5,0.50,0


In [104]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57875
0    0.42125
Name: proportion, dtype: float64

In [105]:
df.industry.mode()
# Ques : 1

0    technology
Name: industry, dtype: str

In [106]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [107]:
cols = df.columns[(df.dtypes != 'str') & (df.columns != 'converted')]
corr_matrix = pd.DataFrame({
    c : df[cols].corrwith(df[c])
    for c in cols
}
)

corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [108]:
corr_pairs = corr_matrix.unstack()

corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
corr_pairs = corr_pairs.drop_duplicates()

sorted_corr_pairs = corr_pairs.sort_values(ascending = False)
print(sorted_corr_pairs)

# Ques : 2

interaction_count         lead_score                  0.915746
lead_score                number_of_courses_viewed    0.757204
number_of_courses_viewed  lead_score                  0.757204
                          interaction_count           0.721609
lead_score                annual_income               0.229496
annual_income             lead_score                  0.229496
                          number_of_courses_viewed    0.161300
                          interaction_count           0.122842
dtype: float64


In [109]:
from sklearn.metrics import mutual_info_score

In [110]:
mutual_info = pd.DataFrame({
    'feature' : categorical,
    'mi_score' : [mutual_info_score(df_full_train[c], df_full_train['converted']) for c in categorical]
})

mutual_info.sort_values(by='mi_score', ascending=False).round({'mi_score': 2})
# max(mutual_info.sort_values(by='mi_score', ascending=False).round({'mi_score': 2})['mi_score'])
# Ques : 3

,feature,mi_score
0,lead_source,0.03
2,employment_status,0.02
1,industry,0.00
3,location,0.00


In [111]:
# although I am not required to use diff and risk ratio for now, but still can use it

global_converted = df_full_train['converted'].mean()

from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).converted.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_converted
    df_group['risk_ratio'] = df_group['mean']/global_converted

    display(df_group)
    print()
    print()

lead_source


,mean,count,diff,risk_ratio
lead_source,,,,
NA,0.579832,119,0.001082,1.001869
events,0.421782,505,-0.156968,0.728781
organic_search,0.679074,994,0.100324,1.173347
paid_ads,0.434879,906,-0.143871,0.751410
referral,0.767145,627,0.188395,1.325521
social_media,0.568905,849,-0.009845,0.982988




industry


,mean,count,diff,risk_ratio
industry,,,,
NA,0.600000,200,0.021250,1.036717
education,0.520161,496,-0.058589,0.898767
finance,0.595782,569,0.017032,1.029429
healthcare,0.562784,661,-0.015966,0.972412
manufacturing,0.567639,377,-0.011111,0.980802
retail,0.556604,742,-0.022146,0.961734
technology,0.627225,955,0.048475,1.083758




employment_status


,mean,count,diff,risk_ratio
employment_status,,,,
NA,0.602564,156,0.023814,1.041147
employed,0.666351,2113,0.087601,1.151363
self_employed,0.553603,569,-0.025147,0.956549
student,0.434564,596,-0.144186,0.750866
unemployed,0.422261,566,-0.156489,0.729609




location


,mean,count,diff,risk_ratio
location,,,,
NA,0.562874,167,-0.015876,0.972569
africa,0.534527,391,-0.044223,0.923589
asia,0.601907,839,0.023157,1.040012
europe,0.600221,903,0.021471,1.037100
north_america,0.579817,1090,0.001067,1.001843
south_america,0.545902,610,-0.032848,0.943243


In [167]:
from sklearn.feature_extraction import DictVectorizer

numerical = list(df.columns[df.dtypes != 'str'])
df[numerical] = df[numerical].fillna(0)

numerical = [c for c in numerical if c != 'converted']

dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical + numerical].to_dict(orient = 'records')

X_train = dv.fit(train_dict)
X_train

DictVectorizer(sparse=False)

In [168]:
X_train = dv.transform(train_dict)
X_train

array([[8.0198e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        5.0000e+00],
       [5.0548e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.6300e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       ...,
       [2.1561e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [3.1691e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        3.0000e+00],
       [4.6342e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        5.0000e+00]], shape=(3000, 28))

In [169]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)
X_val

array([[0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        1.0000e+00],
       [6.3357e+04, 0.0000e+00, 1.0000e+00, ..., 1.0000e+00, 0.0000e+00,
        4.0000e+00],
       [7.2351e+04, 0.0000e+00, 1.0000e+00, ..., 1.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [5.1328e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        3.0000e+00],
       [1.3315e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [5.3098e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        1.0000e+00]], shape=(1000, 28))

In [170]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver = 'liblinear', C = 1.0, max_iter = 1000, random_state = 42)
model.fit(X_train, y_train)



,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [171]:
model.intercept_[0]

np.float64(-0.09288149430633613)

In [172]:
model.coef_[0].round(2)

array([-0.  , -0.  ,  0.03, -0.02, -0.06, -0.05, -0.  , -0.03, -0.01,
       -0.02, -0.01, -0.03,  0.01,  0.25,  0.02, -0.  , -0.04,  0.01,
       -0.07,  0.03, -0.03, -0.  , -0.02, -0.01, -0.01, -0.02, -0.02,
        0.1 ])

In [173]:
y_pred = model.predict_proba(X_val)
y_pred

array([[0.45527705, 0.54472295],
       [0.20994793, 0.79005207],
       [0.49022091, 0.50977909],
       ...,
       [0.24221383, 0.75778617],
       [0.22098616, 0.77901384],
       [0.59553438, 0.40446562]], shape=(1000, 2))

In [174]:
y_pred = y_pred[:, 1]
y_pred

array([0.54472295, 0.79005207, 0.50977909, 0.52913949, 0.82072027,
       0.54246   , 0.50178717, 0.54698398, 0.82553299, 0.40583754,
       0.91301567, 0.33581825, 0.30789883, 0.42704803, 0.6219229 ,
       0.53665458, 0.41931937, 0.84087345, 0.79969125, 0.61790114,
       0.61625181, 0.68059931, 0.85605897, 0.8078487 , 0.62046017,
       0.45970722, 0.54556724, 0.41221393, 0.72714023, 0.46812079,
       0.77222191, 0.79043883, 0.61433884, 0.78549759, 0.49753426,
       0.66077291, 0.69804763, 0.76896543, 0.65657501, 0.64127911,
       0.66509115, 0.57942826, 0.67830946, 0.61186114, 0.86890227,
       0.4146967 , 0.84130129, 0.74229365, 0.68028539, 0.7189143 ,
       0.86417929, 0.64043375, 0.5424699 , 0.70225152, 0.27894903,
       0.70826494, 0.62310718, 0.47781771, 0.37914032, 0.69936035,
       0.52032869, 0.69977698, 0.75148939, 0.58087503, 0.47001802,
       0.56863492, 0.91297185, 0.79972028, 0.82206546, 0.36140197,
       0.66132404, 0.52330065, 0.79295471, 0.72068083, 0.71234

In [175]:
contract_pred = (y_pred >= 0.5)
    
print("Model accuracy:", (y_val == contract_pred).mean())

print("Baseline accuracy:", max(y_val.mean(), 1 - y_val.mean()))

# Ques : 4

Model accuracy: 0.651
Baseline accuracy: 0.586


In [176]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_val, contract_pred))
print(classification_report(y_val, contract_pred))

[[120 294]
 [ 55 531]]
              precision    recall  f1-score   support

           0       0.69      0.29      0.41       414
           1       0.64      0.91      0.75       586

    accuracy                           0.65      1000
   macro avg       0.66      0.60      0.58      1000
weighted avg       0.66      0.65      0.61      1000



In [177]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = contract_pred.astype(int)
df_pred['actual'] = y_val
df_pred['correct'] = (y_val == contract_pred).astype(int)

df_pred.head()

,probability,prediction,actual,correct
0,0.544723,1,0,0
1,0.790052,1,1,1
2,0.509779,1,1,1
3,0.529139,1,1,1
4,0.820720,1,0,0


In [180]:
features = categorical + numerical

original_accuracy = (y_val == contract_pred).mean()

results = {}

for feat in features:
    updated_features = [f for f in features if f != feat]
    train_dict = df_train[updated_features].to_dict(orient='records')
    X_train_new = dv.fit_transform(train_dict)

    model.fit(X_train_new, y_train)

    val_dict = df_val[updated_features].to_dict(orient='records')
    X_val_new = dv.transform(val_dict)
    
    y_pred_new = model.predict_proba(X_val_new)[:, 1]

    accuracy = ((y_pred_new >= 0.5) == y_val).mean()

    results[feat] = {
        'accuracy' : accuracy,
        'difference' : abs(original_accuracy - accuracy)
    }
pd.DataFrame(results).T.sort_values(by = 'difference')

# Ques : 5
    

,accuracy,difference
industry,0.651,0.000
lead_score,0.651,0.000
location,0.650,0.001
lead_source,0.648,0.003
employment_status,0.648,0.003
number_of_courses_viewed,0.645,0.006
interaction_count,0.593,0.058
annual_income,0.716,0.065


In [183]:
C_val = [0.000001, 0.00001, 0.0001, 0.001]
C_best = C_val[0]

accuracy_val = float('-inf')
result = {}

train_dict = df_train[features].to_dict(orient='records')
val_dict = df_val[features].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

for c in C_val:
    model = LogisticRegression(
        solver='liblinear', C=c, max_iter=1000, random_state=42
    )

    model.fit(X_train, y_train)

    y_pred_new_c = model.predict_proba(X_val)[:, 1]

    accuracy = round(((y_pred_new_c >= 0.5) == y_val).mean(), 3)

    result[c] = {
        'C': c,
        'accuracy': accuracy
    }

    if accuracy > accuracy_val:
        C_best = c
        accuracy_val = accuracy

display(pd.DataFrame(result).T.sort_values(by='accuracy', ascending=False))

print("Best C:", C_best)
print("Best accuracy:", accuracy_val)

# Ques : 6

,C,accuracy
0.001000,0.001000,0.648
0.000001,0.000001,0.590
0.000010,0.000010,0.590
0.000100,0.000100,0.590


Best C: 0.001
Best accuracy: 0.648
